# 02 — MSP-Podcast→HCUDB 4クラスdecoder学習・評価

検証済みframe cacheとmanifestだけを入力にし、MSP-Podcast学習、HCUDB継続学習、IEMOCAP外部testを同じ4クラス出力で比較します。正式実行は既定で無効です。


In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.notebook_api import environment_summary, run_demo_transfer_study
from ser_pipeline.study import DatasetArtifacts, run_transfer_study
from ser_pipeline.training import TrainingConfig

RUN_DEMO = True
RUN_FORMAL_STUDY = False
STUDY_SEEDS = (42, 43, 44)
ARTIFACT_DIR = PROJECT_ROOT / 'runs' / 'ser_decoder_study'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 1. cache-only実行環境


In [ ]:
environment_summary()


## 2. 合成cacheでの短時間E2E


In [ ]:
demo_seeds = (42,)
demo_summary = run_demo_transfer_study(ARTIFACT_DIR / 'demo', seeds=demo_seeds, epochs=1) if RUN_DEMO else None
{'status': 'completed' if demo_summary else 'skipped', 'seeds': list(demo_seeds)}


In [ ]:
if demo_summary:
    demo_run = demo_summary['runs'][0]
    rows = []
    for stage_name in ('before', 'after'):
        for dataset_name, payload in demo_run[stage_name].items():
            metrics = payload['result']['metrics_4class']
            rows.append({
                'stage': stage_name,
                'dataset': dataset_name,
                'accuracy_percent': metrics['accuracy'] * 100,
                'uar_percent': metrics['uar'] * 100,
                'macro_f1_percent': metrics['macro_f1'] * 100,
            })
    demo_table = pd.DataFrame(rows)
else:
    demo_table = pd.DataFrame()
demo_table


## 3. 正式3-seed実行ゲート

必要な6つのmanifest/cacheパスを環境変数で与え、全preflight項目の承認後にフラグを変更します。指標ファイルは0–1、ここでの表示だけ百分率です。


In [ ]:
if RUN_FORMAL_STUDY:
    names = ('msp_podcast', 'hcudb1', 'iemocap')
    formal_artifacts = {
        name: DatasetArtifacts(
            manifest_path=Path(os.environ[f'SER_{name.upper()}_MANIFEST']),
            cache_root=Path(os.environ[f'SER_{name.upper()}_CACHE']),
        )
        for name in names
    }
    formal_summary = run_transfer_study(
        formal_artifacts,
        ARTIFACT_DIR / 'formal',
        seeds=STUDY_SEEDS,
        base_config=TrainingConfig(seed=42, device='auto'),
    )
else:
    formal_summary = {'status': 'disabled_by_default', 'seeds': list(STUDY_SEEDS)}
formal_summary
